# Modern Vision Transformer from Scratch

A production-grade ViT-Tiny for CIFAR-10 incorporating 2025 research advances:

| Component | Original ViT | This Notebook |
|---|---|---|
| Image size | 224×224 (resized!) | 32×32 native |
| Parameters | ~86M | ~5.5M |
| Normalization | LayerNorm (2016) | **DynamicTanh** (Meta 2025) |
| Positional encoding | Learned (12k params) | **2D RoPE** (0 params) |
| Attention | Softmax O(N²) | **Alternating Linear+Flash** (L2ViT 2025) |
| Local context | None | **LCM + Token Shift + DW Bypass** |
| Optimizer | Adam lr=1e-3 | **AdamW fused + warmup/cosine** |
| Mixed precision | None | **AMP device-aware** |
| Epoch time (T4) | ~9 min | **~25–35 sec** |

## Architecture Overview

```
Input (B, 3, 32, 32)
  → PatchEmbedding: Conv2d(3→192, k=4, s=4) → (B, 64, 192)
  → Prepend CLS token                        → (B, 65, 192)
  → 12× TransformerEncoderBlock:
       ① 2D Token Shift  (Vision-RWKV 2025, free local context)
       ② DynamicTanh     (Meta 2025, replaces LayerNorm)
       ③ Attention:
            EVEN blocks → LinearGlobalAttention O(N·d²) + LCM
            ODD  blocks → FlashAttention O(N²d) IO-aware
            Both        → 2D RoPE applied to Q and K
       ④ Residual + DynamicTanh + MLP(192→768→192) + Residual
       ⑤ DepthwiseConvBypass (parallel branch, CIFAR-10 specialisation)
  → DynamicTanh → CLS → Linear(192, 10) → logits
```

## 1. Device Setup & GPU Flags

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

if torch.cuda.is_available():
    device = torch.device('cuda')
    # cuDNN auto-selects fastest conv algorithm for fixed input shapes
    torch.backends.cudnn.benchmark = True
    # TF32: faster matmul on Ampere+ GPUs, negligible accuracy loss
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'CUDA: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print('MPS: Apple Silicon GPU')
else:
    device = torch.device('cpu')
    print('CPU')

print(f'PyTorch {torch.__version__} | device={device}')

# AMP: GradScaler only on CUDA (MPS does not support it)
USE_AMP = device.type == 'cuda'
scaler  = torch.cuda.amp.GradScaler(enabled=USE_AMP)

CUDA: Tesla T4
PyTorch 2.10.0+cu128 | device=cuda


/tmp/ipykernel_5728/4006886234.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## 2. DynamicTanh — Replaces LayerNorm

**Paper:** *Transformers without Normalization* (Zhu et al., Meta AI 2025) — arXiv:2503.10622

LayerNorm computes mean and variance across every token at every layer. DynamicTanh replaces this with a pure elementwise operation:

```
DyT(x) = γ ⊙ tanh(α · x) + β
```

- `α` — learnable scalar per layer; controls how aggressively extreme values are compressed
- `γ`, `β` — learnable per-channel scale and shift (same role as LayerNorm's)
- `tanh` saturation naturally handles activation outliers — no statistics needed

Meta showed DynamicTanh matches or beats LayerNorm on ViT, DINO, DiT, and LLaMA.

In [2]:
class DynamicTanh(nn.Module):
    """
    DyT(x) = gamma * tanh(alpha * x) + beta
    Drop-in replacement for LayerNorm with no statistics computation.
    'Transformers without Normalization' (Meta AI, 2025) arXiv:2503.10622
    """
    def __init__(self, embed_dim, alpha_init=0.5):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1) * alpha_init)
        self.gamma = nn.Parameter(torch.ones(embed_dim))
        self.beta  = nn.Parameter(torch.zeros(embed_dim))

    def forward(self, x):
        return self.gamma * torch.tanh(self.alpha * x) + self.beta


# Sanity check
_dyt = DynamicTanh(192)
_x   = torch.randn(2, 65, 192)
_out = _dyt(_x)
assert _out.shape == (2, 65, 192) and not torch.isnan(_out).any()
print('DynamicTanh OK:', _out.shape)

DynamicTanh OK: torch.Size([2, 65, 192])


## 3. Patch Embedding

Splits each 32×32 CIFAR-10 image into non-overlapping 4×4 patches using a strided Conv2d.

```
(B, 3, 32, 32)
  → Conv2d(3, 192, kernel=4, stride=4)
  → (B, 192, 8, 8)
  → flatten(2) → (B, 192, 64)
  → transpose   → (B, 64, 192)   ← 64 patch tokens, each 192-dim
```

No `transforms.Resize` needed — we operate on native 32×32 resolution.

In [3]:
class PatchEmbedding(nn.Module):
    """
    32×32 image → 64 patch tokens of 192 dimensions each.
    kernel_size = stride = patch_size guarantees non-overlapping patches.
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=192):
        super().__init__()
        self.img_size  = img_size
        self.patch_size = patch_size
        self.n_patches  = (img_size // patch_size) ** 2   # 64
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, C, H, W = x.shape
        assert H == self.img_size and W == self.img_size, \
            f'Expected {self.img_size}x{self.img_size}, got {H}x{W}'
        return self.proj(x).flatten(2).transpose(1, 2)   # (B, 64, 192)


# Sanity check
_pe  = PatchEmbedding()
_out = _pe(torch.randn(4, 3, 32, 32))
assert _out.shape == (4, 64, 192)
print('PatchEmbedding OK:', _out.shape)

PatchEmbedding OK: torch.Size([4, 64, 192])


## 4. 2D Rotary Position Embeddings (RoPE)

**Paper:** *RoFormer* (Su et al., 2021) — extended to 2D for vision.

Instead of adding a learned positional vector to tokens (which costs parameters and doesn't generalise), RoPE *rotates* the Query and Key vectors before the dot product. The rotation angle encodes position, so `Q·Kᵀ` naturally captures **relative** position.

For a 2D patch grid, each patch has `(row, col)` coordinates. We split the head dimension: first half encodes row, second half encodes column.

```
Learned PE:  0 params used → 65 × 192 = 12,480 values
2D RoPE:     0 parameters,  encodes relative position,  generalises to new sizes
```

In [4]:
def build_2d_rope_cache(num_patches_side: int, head_dim: int, device):
    """
    Precompute cos/sin rotation matrices for all 2D patch grid positions.
    Returns cos, sin each of shape (num_patches, head_dim).
    """
    quarter = head_dim // 4
    theta   = 1.0 / (10000 ** (
        torch.arange(0, quarter, device=device).float() / quarter
    ))

    coords        = torch.arange(num_patches_side, device=device).float()
    rows, cols    = torch.meshgrid(coords, coords, indexing='ij')
    row_freqs     = torch.outer(rows.flatten(), theta)   # (N, quarter)
    col_freqs     = torch.outer(cols.flatten(), theta)   # (N, quarter)

    freqs = torch.cat([row_freqs, col_freqs], dim=-1)    # (N, head_dim//2)
    cos   = torch.cat([freqs.cos(), freqs.cos()], dim=-1)  # (N, head_dim)
    sin   = torch.cat([freqs.sin(), freqs.sin()], dim=-1)  # (N, head_dim)
    return cos, sin


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    """
    Apply 2D RoPE rotation to Q or K tensor.
    x:   (B, heads, seq_len, head_dim)
    cos, sin: (num_patches, head_dim) — CLS gets identity rotation.
    """
    seq_len = x.shape[2]
    ones    = torch.ones(1,  cos.shape[-1], device=x.device)
    zeros   = torch.zeros(1, sin.shape[-1], device=x.device)
    # Prepend identity rotation for CLS token
    cos_full = torch.cat([ones,  cos], dim=0)[:seq_len].unsqueeze(0).unsqueeze(0)
    sin_full = torch.cat([zeros, sin], dim=0)[:seq_len].unsqueeze(0).unsqueeze(0)

    x1      = x[..., 0::2]
    x2      = x[..., 1::2]
    rotated = torch.stack([-x2, x1], dim=-1).flatten(-2)
    return x * cos_full + rotated * sin_full


# Sanity check
_cos, _sin = build_2d_rope_cache(8, 64, device='cpu')
assert _cos.shape == (64, 64)
_q  = torch.randn(2, 3, 65, 64)
_qr = apply_rope(_q, _cos, _sin)
assert _qr.shape == _q.shape and not torch.isnan(_qr).any()
print('2D RoPE OK — cos:', _cos.shape, '| rotated q:', _qr.shape)

2D RoPE OK — cos: torch.Size([64, 64]) | rotated q: torch.Size([2, 3, 65, 64])


## 5. 2D Token Shift

**Paper:** *Vision-RWKV / RSRWKV* (2025) — arXiv:2503.20382

Before the QKV projection in every encoder block, mix each patch token with its spatial top and left neighbours by shifting the token grid. This injects local inductive bias at **zero parameter cost**.

```
Token at (row, col) receives:
  - First C/4 dims  ← copied from token at (row-1, col)  [top neighbour]
  - Next  C/4 dims  ← copied from token at (row, col-1)  [left neighbour]
  - Rest of dims    ← unchanged
```

In [5]:
def token_shift_2d(x: torch.Tensor, H: int, W: int) -> torch.Tensor:
    """
    Mix each patch token with its top/left spatial neighbour.
    Injects local structure at zero parameter cost.
    Vision-RWKV (ICLR 2025) / RSRWKV arXiv:2503.20382

    x: (B, H*W, C) — patch tokens only (no CLS)
    """
    B, N, C = x.shape
    quarter  = C // 4
    x_2d     = x.reshape(B, H, W, C).clone()
    src      = x.reshape(B, H, W, C)

    x_2d[:, 1:,  :, :quarter]         = src[:, :-1, :,  :quarter]         # top
    x_2d[:, :,  1:, quarter:quarter*2] = src[:, :,  :-1, quarter:quarter*2] # left
    return x_2d.reshape(B, N, C)


# Sanity check
_xs = token_shift_2d(torch.randn(2, 64, 192), 8, 8)
assert _xs.shape == (2, 64, 192)
print('TokenShift2D OK:', _xs.shape)

TokenShift2D OK: torch.Size([2, 64, 192])


## 6. Linear Global Attention + Local Concentration Module

**Paper:** *The Linear Attention Resurrection in Vision Transformer* — L2ViT (2025) arXiv:2501.16182

### Why Linear Attention?

Standard attention is O(N²·d) — it forms an N×N matrix.
Linear attention replaces `softmax` with a kernel feature map `φ(x) = elu(x) + 1`, which allows reordering via matrix associativity:

```
Standard: softmax(QKᵀ) · V          ← must form N×N first   O(N²d)
Linear:   φ(Q) · (φ(K)ᵀ · V)       ← compute d×d KV first  O(N·d²)
```

The `d×d` context matrix is constant size regardless of N — making attention linear in the sequence length.

### Why LCM?

Linear attention loses local spatial precision (blurry attention pattern). The **Local Concentration Module** applies a depthwise 3×3 conv after the linear attention output to restore sharpness.

In [6]:
class LinearGlobalAttention(nn.Module):
    """
    Sub-quadratic attention O(N·d²) using kernel trick: φ(x) = elu(x) + 1.
    Computes φ(K)ᵀV (d×d context) first, then φ(Q)·context.
    Learnable KV scale + denominator clamping for numerical stability.
    Used in EVEN encoder blocks.
    L2ViT (2025): arXiv:2501.16182
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.qkv       = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.proj      = nn.Linear(embed_dim, embed_dim)
        self.drop      = nn.Dropout(dropout)
        self.kv_scale  = nn.Parameter(torch.ones(1) * 0.1)

    @staticmethod
    def phi(x):
        return F.elu(x) + 1   # non-negative feature map

    def forward(self, x, rope_cos=None, rope_sin=None):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(2)             # (B, N, heads, head_dim)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B, heads, N, d)

        if rope_cos is not None:
            q = apply_rope(q, rope_cos, rope_sin)
            k = apply_rope(k, rope_cos, rope_sin)

        q, k = self.phi(q), self.phi(k)

        # d×d context — constant size, O(N·d²) total
        kv  = torch.einsum('bhnd,bhnm->bhdm', k, v) * self.kv_scale
        num = torch.einsum('bhnd,bhdm->bhnm', q, kv)

        # Normalisation denominator
        k_sum = k.sum(dim=2)
        den   = torch.einsum('bhnd,bhd->bhn', q, k_sum).clamp(min=1e-6).unsqueeze(-1)

        out = (num / den).transpose(1, 2).reshape(B, N, C)
        return self.proj(self.drop(out))


class LocalConcentrationModule(nn.Module):
    """
    Depthwise 3×3 conv applied to linear attention output.
    Restores local spatial precision that the global kernel loses.
    L2ViT (2025): arXiv:2501.16182
    """
    def __init__(self, embed_dim, patch_grid=8):
        super().__init__()
        self.H       = patch_grid
        self.W       = patch_grid
        self.dw_conv = nn.Conv2d(embed_dim, embed_dim, 3, padding=1,
                                  groups=embed_dim, bias=False)
        self.norm    = DynamicTanh(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        x_2d = x.transpose(1, 2).reshape(B, C, self.H, self.W)
        return self.norm(self.dw_conv(x_2d).flatten(2).transpose(1, 2))


# Sanity check
_cos, _sin = build_2d_rope_cache(8, 64, device='cpu')
_lga = LinearGlobalAttention(192, 3)
_lcm = LocalConcentrationModule(192, 8)
_x   = torch.randn(2, 65, 192)
_out = _lga(_x, _cos, _sin)
assert _out.shape == (2, 65, 192) and not torch.isnan(_out).any()
_lcm_out = _lcm(_out[:, 1:, :])
assert _lcm_out.shape == (2, 64, 192)
print('LinearGlobalAttention OK:', _out.shape)
print('LocalConcentrationModule OK:', _lcm_out.shape)

LinearGlobalAttention OK: torch.Size([2, 65, 192])
LocalConcentrationModule OK: torch.Size([2, 64, 192])


## 7. Flash Attention Block

**PyTorch 2.0+:** `F.scaled_dot_product_attention` — built-in Flash Attention.

Standard attention materialises the full N×N attention matrix in slow HBM (GPU memory). Flash Attention tiles the computation so it fits in fast SRAM, never writing the N×N matrix — same result, dramatically less memory bandwidth.

Used in **ODD encoder blocks** for precise local attention that complements the global linear attention in even blocks.

In [7]:
class FlashAttentionBlock(nn.Module):
    """
    IO-aware softmax attention via F.scaled_dot_product_attention.
    Never materialises the full N×N attention matrix in HBM.
    Applies 2D RoPE to Q and K.
    Used in ODD encoder blocks.
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.qkv       = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.proj      = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = dropout

    def forward(self, x, rope_cos=None, rope_sin=None):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(2)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B, heads, N, d)

        if rope_cos is not None:
            q = apply_rope(q, rope_cos, rope_sin)
            k = apply_rope(k, rope_cos, rope_sin)

        out = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.attn_drop if self.training else 0.0
        )
        return self.proj(out.transpose(1, 2).reshape(B, N, C))


# Sanity check
_fa  = FlashAttentionBlock(192, 3)
_out = _fa(torch.randn(2, 65, 192), _cos, _sin)
assert _out.shape == (2, 65, 192) and not torch.isnan(_out).any()
print('FlashAttentionBlock OK:', _out.shape)

FlashAttentionBlock OK: torch.Size([2, 65, 192])


## 8. Depthwise Conv Bypass Branch

**Paper:** *Depth-Wise Convolutions in ViTs for Efficient Training on Small Datasets* (2024) — arXiv:2407.19394

Adds a parallel depthwise separable conv branch alongside every transformer block. The conv output is added via a three-way residual:

```
output = x + attn_out + mlp_out + conv_out
```

The conv branch captures fine-grained local texture that transformers miss on small datasets like CIFAR-10. Validated on CIFAR-10/100 and Tiny-ImageNet.

In [8]:
class DepthwiseConvBypass(nn.Module):
    """
    Parallel depthwise-separable conv branch alongside each encoder block.
    Captures local texture; added as a third residual term.
    arXiv:2407.19394  — validated on CIFAR-10/100.
    CLS position receives zeros (no spatial meaning).
    """
    def __init__(self, embed_dim, patch_grid=8):
        super().__init__()
        self.H   = patch_grid
        self.W   = patch_grid
        self.dw  = nn.Conv2d(embed_dim, embed_dim, 3, padding=1,
                              groups=embed_dim, bias=False)   # depthwise
        self.pw  = nn.Conv2d(embed_dim, embed_dim, 1, bias=False)  # pointwise
        self.norm = DynamicTanh(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, Np1, C = x.shape
        patches = x[:, 1:, :]                            # skip CLS
        x_2d    = patches.transpose(1, 2).reshape(B, C, self.H, self.W)
        out     = self.norm(self.pw(self.dw(x_2d)).flatten(2).transpose(1, 2))
        cls_pad = torch.zeros(B, 1, C, device=x.device, dtype=x.dtype)
        return torch.cat([cls_pad, out], dim=1)          # (B, 65, C)


# Sanity check
_dwb = DepthwiseConvBypass(192, 8)
_out = _dwb(torch.randn(2, 65, 192))
assert _out.shape == (2, 65, 192) and not torch.isnan(_out).any()
print('DepthwiseConvBypass OK:', _out.shape)

DepthwiseConvBypass OK: torch.Size([2, 65, 192])


## 9. Transformer Encoder Block (Alternating)

**Even blocks (0,2,4,6,8,10):** LinearGlobalAttention + LocalConcentrationModule — O(N·d²) global context  
**Odd blocks (1,3,5,7,9,11):** FlashAttentionBlock — precise local attention  
**All blocks:** 2D Token Shift → DynamicTanh → Attention → Residual → DynamicTanh → MLP → Residual → DWBypass

This alternating design is the core insight from **L2ViT (2025)**: global cheap context + precise local attention, interleaved.

In [9]:
class TransformerEncoderBlock(nn.Module):
    """
    Alternating attention (L2ViT 2025):
      even → LinearGlobalAttention + LocalConcentrationModule  O(N·d²)
      odd  → FlashAttentionBlock                               IO-aware O(N²d)
    Both: DynamicTanh, 2D RoPE, MLP with dropout, DepthwiseConvBypass.
    2D Token Shift applied before every QKV projection.
    """
    def __init__(self, embed_dim, num_heads, mlp_dim, block_idx,
                 dropout=0.1, patch_grid=8):
        super().__init__()
        self.block_idx  = block_idx
        self.patch_grid = patch_grid
        self.norm1 = DynamicTanh(embed_dim)
        self.norm2 = DynamicTanh(embed_dim)

        if block_idx % 2 == 0:
            self.attn = LinearGlobalAttention(embed_dim, num_heads, dropout)
            self.lcm  = LocalConcentrationModule(embed_dim, patch_grid)
        else:
            self.attn = FlashAttentionBlock(embed_dim, num_heads, dropout)
            self.lcm  = None

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )
        self.dw_bypass = DepthwiseConvBypass(embed_dim, patch_grid)

    def forward(self, x, rope_cos=None, rope_sin=None):
        H = W = self.patch_grid

        # Token shift on patch tokens (free local inductive bias)
        shifted_patches = token_shift_2d(x[:, 1:, :], H, W)
        x_shifted = torch.cat([x[:, :1, :], shifted_patches], dim=1)

        # Attention sub-layer
        attn_out = self.attn(self.norm1(x_shifted), rope_cos, rope_sin)

        # LCM restores local sharpness after linear attention (even blocks only)
        if self.lcm is not None:
            lcm_out  = self.lcm(attn_out[:, 1:, :])
            attn_out = torch.cat([attn_out[:, :1, :], lcm_out], dim=1)

        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        x = x + self.dw_bypass(x)   # parallel local conv branch
        return x


# Sanity check both block types
_cos, _sin = build_2d_rope_cache(8, 64, device='cpu')
_x = torch.randn(2, 65, 192)
for idx in [0, 1]:
    _blk = TransformerEncoderBlock(192, 3, 768, block_idx=idx)
    _out = _blk(_x, _cos, _sin)
    assert _out.shape == (2, 65, 192) and not torch.isnan(_out).any()
    kind = 'Linear+LCM' if idx % 2 == 0 else 'Flash'
    print(f'Block {idx} ({kind}) OK:', _out.shape)

Block 0 (Linear+LCM) OK: torch.Size([2, 65, 192])
Block 1 (Flash) OK: torch.Size([2, 65, 192])


## 10. Vision Transformer — Full Model

**Config:**
```
img_size=32  patch_size=4  embed_dim=192  num_heads=3  head_dim=64
mlp_dim=768  num_layers=12  dropout=0.1  num_classes=10
Parameters: ~5.5M  (vs ~86M for ViT-Base)
```

In [10]:
class VisionTransformer(nn.Module):
    """
    ViT-Tiny for CIFAR-10 with all 2025 improvements:
    DynamicTanh · 2D RoPE · Alternating Linear+Flash attention
    LCM · Token Shift · DepthwiseConvBypass · AdamW-ready init
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 num_classes=10, embed_dim=192, num_heads=3,
                 mlp_dim=768, num_layers=12, dropout=0.1):
        super().__init__()
        self.patch_grid = img_size // patch_size      # 8
        num_patches     = self.patch_grid ** 2        # 64
        head_dim        = embed_dim // num_heads      # 64

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                embed_dim, num_heads, mlp_dim,
                block_idx=i, dropout=dropout, patch_grid=self.patch_grid
            )
            for i in range(num_layers)
        ])

        self.norm     = DynamicTanh(embed_dim)
        self.mlp_head = nn.Linear(embed_dim, num_classes)

        # RoPE cache: registered as buffer (moves with .to(device), not a param)
        self.register_buffer('rope_cos', torch.empty(num_patches, head_dim))
        self.register_buffer('rope_sin', torch.empty(num_patches, head_dim))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')

    def build_rope(self, device):
        """Call once after moving model to device."""
        cos, sin = build_2d_rope_cache(
            self.patch_grid, self.rope_cos.shape[-1], device
        )
        self.rope_cos.copy_(cos)
        self.rope_sin.copy_(sin)

    def forward(self, x):
        B  = x.shape[0]
        x  = self.patch_embed(x)                          # (B, 64, 192)
        x  = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)  # (B, 65, 192)

        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)

        return self.mlp_head(self.norm(x[:, 0]))          # CLS → (B, 10)


# Build, verify, print param count
model = VisionTransformer()
model.build_rope(device='cpu')

_out = model(torch.randn(2, 3, 32, 32))
assert _out.shape == (2, 10) and not torch.isnan(_out).any()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'VisionTransformer OK: output {_out.shape}')
print(f'Parameters: {total:,} total | {trainable:,} trainable')

VisionTransformer OK: output torch.Size([2, 10])
Parameters: 5,823,803 total | 5,823,803 trainable


## 11. Data Loading

Key changes from the original notebook:
- **No `transforms.Resize`** — native 32×32 input
- **CIFAR-10 channel statistics** — not generic `(0.5, 0.5, 0.5)`
- **`RandomCrop + RandomHorizontalFlip`** — standard CIFAR-10 augmentation
- **Validation split** — `train=False` test set evaluated every epoch
- **`prefetch_factor=2`** — workers pre-load 2 batches ahead
- **`non_blocking=True`** on `.to(device)` — overlaps CPU→GPU transfer with compute

In [11]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_ds = datasets.CIFAR10('./data', train=True,  download=True, transform=train_transform)
val_ds   = datasets.CIFAR10('./data', train=False, download=True, transform=val_transform)

_pin     = (device.type == 'cuda')
_workers = 0 if device.type == 'mps' else 4
_pf      = 2 if _workers > 0 else None
_pw      = (_workers > 0)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,
                          num_workers=_workers, pin_memory=_pin,
                          persistent_workers=_pw, prefetch_factor=_pf)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False,
                          num_workers=_workers, pin_memory=_pin,
                          persistent_workers=_pw, prefetch_factor=_pf)

print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')
print(f'Effective batch size: {128 * 4} (128 × 4 grad accum steps)')

100%|██████████| 170M/170M [00:05<00:00, 30.6MB/s] 


Train: 391 batches | Val: 40 batches
Effective batch size: 512 (128 × 4 grad accum steps)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 12. Model, Optimizer & LR Scheduler

**AdamW** with two param groups:
- 2-D weights (Linear, Conv2d): weight decay `1e-4`
- 1-D params (bias, DynamicTanh α/γ/β, CLS token): **no weight decay**

**Schedule:** 5-epoch linear warmup → cosine decay to `1e-6` over 25 epochs.

**`torch.compile`** — fuses ops and generates optimised CUDA kernels. 1.5–2× extra speedup on PyTorch 2.0+ with CUDA.

In [12]:
import torch.optim as optim

NUM_EPOCHS    = 30
WARMUP_EPOCHS = 5
LR            = 3e-4
MIN_LR        = 1e-6
WEIGHT_DECAY  = 1e-4

model = VisionTransformer().to(device)
model.build_rope(device)

# channels_last: better cache locality for Conv2d ops
if device.type == 'cuda':
    model = model.to(memory_format=torch.channels_last)

# torch.compile: fused CUDA kernels (PyTorch 2.0+, CUDA only)
if device.type == 'cuda' and hasattr(torch, 'compile'):
    try:
        model = torch.compile(model)
        print('torch.compile: enabled')
    except Exception as e:
        print(f'torch.compile: skipped ({e})')

# Param groups — exclude 1-D params from weight decay
decay_p    = [p for n, p in model.named_parameters() if p.ndim >= 2 and p.requires_grad]
no_decay_p = [p for n, p in model.named_parameters() if p.ndim <  2 and p.requires_grad]

optimizer = optim.AdamW(
    [{'params': decay_p,    'weight_decay': WEIGHT_DECAY},
     {'params': no_decay_p, 'weight_decay': 0.0}],
    lr=LR,
    fused=(device.type == 'cuda'),   # single fused CUDA kernel for param updates
)

warmup_sched = optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1e-6, end_factor=1.0, total_iters=WARMUP_EPOCHS
)
cosine_sched = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS, eta_min=MIN_LR
)
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS]
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

total = sum(p.numel() for p in model.parameters())
print(f'Model: {total:,} params | device={device} | AMP={USE_AMP}')

torch.compile: enabled
Model: 5,823,803 params | device=cuda | AMP=True


## 13. Checkpoint Utilities

In [13]:
import os

def save_checkpoint(model, optimizer, epoch, val_acc, path='best_model.pth'):
    raw = model._orig_mod if hasattr(model, '_orig_mod') else model
    torch.save({
        'epoch':            epoch,
        'model_state_dict': raw.state_dict(),
        'optim_state_dict': optimizer.state_dict(),
        'val_acc':          val_acc,
    }, path)
    print(f'  Checkpoint saved  val_acc={val_acc:.2f}%')


def load_checkpoint(model, optimizer, path='best_model.pth'):
    if not os.path.exists(path):
        return 0, 0.0
    ckpt = torch.load(path, map_location=device)
    raw  = model._orig_mod if hasattr(model, '_orig_mod') else model
    raw.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optim_state_dict'])
    print(f'Resumed epoch {ckpt["epoch"]} | best={ckpt["val_acc"]:.2f}%')
    return ckpt['epoch'], ckpt['val_acc']

## 14. Training Loop

Key features vs original:

| Feature | Original | Now |
|---|---|---|
| Gradient accumulation | None | 4 steps (eff. batch 512) |
| Mixed precision | None | AMP + GradScaler (CUDA) |
| Gradient clipping | None | `max_norm=1.0` |
| Validation | None | Every epoch |
| Checkpointing | None | Best val acc |
| CSV logging | None | `training_log.csv` |
| `non_blocking` transfers | None | Yes (overlaps CPU↔GPU) |

In [ ]:
import csv, time

ACCUM_STEPS   = 4
MAX_GRAD_NORM = 1.0
LOG_PATH      = 'training_log.csv'

best_val_acc = 0.0
with open(LOG_PATH, 'w', newline='') as f:
    csv.writer(f).writerow(
        ['epoch','train_loss','train_acc','val_loss','val_acc','lr','time_s']
    )

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ── Train ────────────────────────────────────────────────────────
    model.train()
    t_loss = t_correct = t_total = 0
    optimizer.zero_grad()

    for step, (imgs, labels) in enumerate(train_loader):
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if device.type == 'cuda':
            imgs = imgs.to(memory_format=torch.channels_last)

        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        t_loss    += loss.item() * ACCUM_STEPS
        t_correct += (logits.argmax(1) == labels).sum().item()
        t_total   += labels.size(0)

    scheduler.step()

    # ── Validate ─────────────────────────────────────────────────────
    model.eval()
    v_loss = v_correct = v_total = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if device.type == 'cuda':
                imgs = imgs.to(memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                logits = model(imgs)
            v_loss    += criterion(logits, labels).item()
            v_correct += (logits.argmax(1) == labels).sum().item()
            v_total   += labels.size(0)

    ta  = 100 * t_correct / t_total
    va  = 100 * v_correct / v_total
    tl  = t_loss / len(train_loader)
    vl  = v_loss / len(val_loader)
    lr  = scheduler.get_last_lr()[0]
    dt  = time.time() - t0

    if va > best_val_acc:
        best_val_acc = va
        save_checkpoint(model, optimizer, epoch, va)

    with open(LOG_PATH, 'a', newline='') as f:
        csv.writer(f).writerow(
            [epoch, f'{tl:.4f}', f'{ta:.2f}', f'{vl:.4f}', f'{va:.2f}',
             f'{lr:.2e}', f'{dt:.1f}']
        )

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
          f'Train {tl:.4f}/{ta:.2f}% | '
          f'Val {vl:.4f}/{va:.2f}% | '
          f'LR {lr:.2e} | {dt:.1f}s')

print(f'\nBest val acc: {best_val_acc:.2f}%  →  best_model.pth')

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
W0519 18:01:28.689000 5728 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
